# NB08 — Clustering de Zonas Industriales por Perfil de Riesgo Vial
**ZMM Movilidad Predictiva**

**Objetivo:** Crear tipologías de zonas industriales basadas en siniestralidad real (Haversine 2km), no en datos municipales agregados.

| Input | Output |
|-------|--------|
| `catalogo_zonas_industriales.csv` (183 zonas) | `zonas_clusterizadas.csv` |
| `rativ_unificado.csv` (203,890 siniestros) | `mapa_zonas_clusters.html` |
| | `nb08_seleccion_k_zonas.png` |

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

RUTA_PROCESSED = '../data_processed/'
RUTA_OUTPUTS   = '../outputs/'

print('='*65)
print('NB08: CLUSTERING DE ZONAS POR PERFIL DE RIESGO VIAL')
print('='*65)

## 1. Carga de Datos

In [ ]:
# --- Catálogo de zonas ---
zonas = pd.read_csv(RUTA_PROCESSED + 'catalogo_zonas_industriales.csv')
print(f'Zonas industriales: {len(zonas)}')
print(f'Columnas: {zonas.columns.tolist()}')

# --- Siniestros RATIV ---
rativ = pd.read_csv(RUTA_PROCESSED + 'rativ_unificado.csv', low_memory=False)
print(f'\nSiniestros RATIV: {len(rativ):,}')

# Parsear coordenadas desde columna Referencia
PATRON_COORD = re.compile(r'^(-?\d{1,3}\.\d+),\s*(-?\d{1,3}\.\d+)$')

def extraer_coords(val):
    if pd.isna(val):
        return np.nan, np.nan
    m = PATRON_COORD.match(str(val).strip())
    if m:
        return float(m.group(1)), float(m.group(2))
    return np.nan, np.nan

rativ[['lat', 'lon']] = rativ['Referencia'].apply(
    lambda v: pd.Series(extraer_coords(v))
)

# Bounding box ZMM
LAT_MIN, LAT_MAX = 25.4, 26.1
LON_MIN, LON_MAX = -100.7, -99.8

rativ_geo = rativ[
    rativ['lat'].between(LAT_MIN, LAT_MAX) &
    rativ['lon'].between(LON_MIN, LON_MAX)
].copy()

print(f'Siniestros con coordenadas válidas en ZMM: {len(rativ_geo):,}')

# Parsear hora de reporte (manejo robusto de SD/99/inválidos)
def extraer_hora(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ('SD', 'sd', '99', '') or ':' not in s:
        return np.nan
    try:
        return int(s.split(':')[0])
    except (ValueError, IndexError):
        return np.nan

rativ_geo['hora'] = rativ_geo['Hora de reporte'].apply(extraer_hora)

# Parsear día de la semana desde fecha
col_dia = [c for c in rativ_geo.columns if 'a' in c.lower() and c.startswith('D')][0]  # 'Día'
col_mes = 'Mes'
col_anio = [c for c in rativ_geo.columns if 'o' in c.lower() and c.startswith('A')][0]  # 'Año'

rativ_geo['fecha'] = pd.to_datetime(
    rativ_geo[[col_anio, col_mes, col_dia]].rename(
        columns={col_anio: 'year', col_mes: 'month', col_dia: 'day'}
    ), errors='coerce'
)
rativ_geo['dia_semana'] = rativ_geo['fecha'].dt.dayofweek  # 0=lun
rativ_geo['es_finde'] = rativ_geo['dia_semana'].isin([5, 6]).astype(int)

hora_ok = rativ_geo['hora'].notna().sum()
print(f'Siniestros con hora válida: {hora_ok:,} ({hora_ok/len(rativ_geo)*100:.1f}%)')
print(f'Siniestros sin hora (SD/99): {len(rativ_geo)-hora_ok:,}')

# CHECKPOINT 1
assert len(zonas) == 183
assert len(rativ_geo) > 100_000
print('\n✓ CHECKPOINT 1: Datos cargados correctamente')

## 2. Asignación Siniestro → Zona (BallTree Haversine, radio 2 km)

In [ ]:
# Coordenadas de zonas y siniestros en radianes
zonas_rad = np.radians(zonas[['latitud', 'longitud']].values)
sin_rad   = np.radians(rativ_geo[['lat', 'lon']].values)

# BallTree con métrica haversine
tree = BallTree(zonas_rad, metric='haversine')

# Query: zona más cercana para cada siniestro
dist_rad, idx = tree.query(sin_rad, k=1)
dist_m = dist_rad.flatten() * 6_371_000  # radio Tierra → metros

rativ_geo = rativ_geo.copy()
rativ_geo['zona_id']     = idx.flatten()
rativ_geo['dist_zona_m'] = dist_m
rativ_geo['en_zona_2km'] = (dist_m <= 2000).astype(int)

dentro = rativ_geo['en_zona_2km'].sum()
print(f'Siniestros dentro de 2 km de alguna zona: {dentro:,} ({dentro/len(rativ_geo)*100:.1f}%)')
print(f'Siniestros fuera de 2 km (descartados):   {len(rativ_geo)-dentro:,}')

# Filtrar solo los que están dentro
rativ_zona = rativ_geo[rativ_geo['en_zona_2km'] == 1].copy()
print(f'\n✓ CHECKPOINT 2: {len(rativ_zona):,} siniestros asignados a zonas')

## 3. Agregación por Zona: Features de Riesgo Vial

In [ ]:
# Horas pico según EDA: 7-9h, 14-15h, 18-20h
HORAS_PICO      = {7, 8, 9, 14, 15, 18, 19, 20}
HORAS_MADRUGADA = {0, 1, 2, 3, 4, 5}

agg_list = []
for zona_idx in range(len(zonas)):
    sz = rativ_zona[rativ_zona['zona_id'] == zona_idx]
    total = len(sz)

    if total > 0:
        hora_pico   = sz[sz['hora'].isin(HORAS_PICO)].shape[0]
        madrugada   = sz[sz['hora'].isin(HORAS_MADRUGADA)].shape[0]
        finde       = sz[sz['es_finde'] == 1].shape[0]
        lesionados  = sz['Total de lesionados'].sum()
        fallecidos  = sz['Total de fallecidos'].sum()
        ratio_les   = lesionados / total
        ratio_fall  = fallecidos / total
    else:
        hora_pico = madrugada = finde = 0
        ratio_les = ratio_fall = 0.0

    row = zonas.iloc[zona_idx]
    agg_list.append({
        'zona_id':               zona_idx,
        'nombre_zona':           row['nombre_zona'],
        'municipio':             row['municipio'],
        'tipo_zona':             row['tipo_zona'],
        'latitud':               row['latitud'],
        'longitud':              row['longitud'],
        'num_empresas':          row['num_empresas'],
        'radio_aprox_m':         row['radio_aprox_m'],
        'total_siniestros':      total,
        'siniestros_hora_pico':  hora_pico,
        'siniestros_madrugada':  madrugada,
        'siniestros_finde':      finde,
        'ratio_lesionados':      ratio_les,
        'ratio_fallecidos':      ratio_fall,
    })

df_zonas = pd.DataFrame(agg_list)

print(f'Zonas con ≥1 siniestro: {(df_zonas["total_siniestros"]>0).sum()}')
print(f'Zonas con 0 siniestros:  {(df_zonas["total_siniestros"]==0).sum()}')
print(f'\nDistribución total_siniestros:')
print(df_zonas['total_siniestros'].describe().round(1))
print(f'\n✓ CHECKPOINT 3: Features de riesgo calculadas para {len(df_zonas)} zonas')

## 4. Filtrado, Transformación y Escalado

In [ ]:
# Filtrar zonas con <5 siniestros (ruido estadístico)
antes = len(df_zonas)
df_valid = df_zonas[df_zonas['total_siniestros'] >= 5].copy()
print(f'Zonas antes del filtro: {antes}')
print(f'Zonas después (≥5 sin): {len(df_valid)} (eliminadas: {antes - len(df_valid)})')

# Codificar tipo_zona
tipo_map = {'A_zona_densa': 2, 'B_ancla_manufactura': 1, 'C_ancla_transporte': 0}
df_valid['tipo_zona_cod'] = df_valid['tipo_zona'].map(tipo_map).fillna(1)

# log1p para variables con cola larga
for col in ['total_siniestros', 'siniestros_hora_pico', 'siniestros_madrugada',
            'siniestros_finde', 'num_empresas']:
    df_valid[f'log_{col}'] = np.log1p(df_valid[col])

# Features para clustering: perfil de riesgo vial
FEATURES_CLUSTER = [
    'log_total_siniestros',
    'log_siniestros_hora_pico',
    'log_siniestros_madrugada',
    'log_siniestros_finde',
    'ratio_lesionados',
    'ratio_fallecidos',
    'tipo_zona_cod',
    'log_num_empresas',
]

X = df_valid[FEATURES_CLUSTER].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nMatriz de clustering: {X_scaled.shape}')
print(f'Features: {FEATURES_CLUSTER}')
print(f'Media post-escala (≈0): {np.abs(X_scaled.mean(axis=0)).max():.6f}')
print(f'\n✓ CHECKPOINT 4: Datos listos para K-Means')

## 5. Selección de K (Silhouette + Interpretabilidad)

In [ ]:
k_range = range(3, 8)
resultados = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    counts = pd.Series(labels).value_counts()
    min_size = counts.min()
    degenerados = (counts < 5).sum()
    resultados.append({'k': k, 'silhouette': sil, 'inercia': km.inertia_,
                       'min_cluster': min_size, 'degenerados': degenerados})
    print(f'K={k}: sil={sil:.4f} | inercia={km.inertia_:.0f} | '
          f'min_cluster={min_size} | degenerados={degenerados}')

res_df = pd.DataFrame(resultados)
# Score: silhouette penalizado por clusters degenerados
res_df['score'] = res_df['silhouette'] - res_df['degenerados'] * 0.1
mejor_k = int(res_df.loc[res_df['score'].idxmax(), 'k'])
print(f'\nMejor K (score): {mejor_k}')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(res_df['k'], res_df['inercia'], 'o-', color='steelblue', lw=2, ms=8)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

axes[1].plot(res_df['k'], res_df['silhouette'], 'o-', color='coral', lw=2, ms=8)
axes[1].axvline(x=mejor_k, color='red', ls='--', alpha=0.5, label=f'K={mejor_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUTA_OUTPUTS + 'nb08_seleccion_k_zonas.png', dpi=150)
plt.show()
print(f'✓ Gráfica guardada')

## 6. Modelo Final y Caracterización de Tipologías

In [ ]:
K_FINAL = mejor_k
print(f'Entrenando K-Means final con K={K_FINAL}...')

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df_valid['cluster_zona'] = km_final.fit_predict(X_scaled)

# Distribución
print(f'\nDistribución de clusters:')
distrib = df_valid['cluster_zona'].value_counts().sort_index()
for c, n in distrib.items():
    print(f'  Cluster {c}: {n} zonas ({n/len(df_valid)*100:.1f}%)')

# Caracterización detallada
print(f'\n{"="*65}')
print('TIPOLOGÍAS DE ZONAS INDUSTRIALES')
print(f'{"="*65}')

for cid in sorted(df_valid['cluster_zona'].unique()):
    zc = df_valid[df_valid['cluster_zona'] == cid]
    n = len(zc)
    print(f'\n--- CLUSTER {cid} ({n} zonas, {n/len(df_valid)*100:.1f}%) ---')
    print(f'  Siniestros totales (media): {zc["total_siniestros"].mean():.0f}')
    print(f'  Siniestros hora pico (media): {zc["siniestros_hora_pico"].mean():.0f}')
    print(f'  Siniestros madrugada (media): {zc["siniestros_madrugada"].mean():.0f}')
    print(f'  Siniestros finde (media): {zc["siniestros_finde"].mean():.0f}')
    print(f'  Ratio lesionados: {zc["ratio_lesionados"].mean():.3f}')
    print(f'  Ratio fallecidos: {zc["ratio_fallecidos"].mean():.4f}')
    print(f'  Empresas (media): {zc["num_empresas"].mean():.1f}')
    mun_top = zc['municipio'].value_counts()
    print(f'  Municipio principal: {mun_top.index[0]} ({mun_top.iloc[0]/n*100:.0f}%)')
    tipo_top = zc['tipo_zona'].value_counts()
    print(f'  Tipo dominante: {tipo_top.index[0]}')

    # 3 zonas más cercanas al centroide (representativas reales)
    centroid = X_scaled[df_valid['cluster_zona'].values == cid].mean(axis=0)
    dists_c = np.linalg.norm(
        X_scaled[df_valid['cluster_zona'].values == cid] - centroid, axis=1
    )
    top3_idx = np.argsort(dists_c)[:3]
    reps = zc.iloc[top3_idx]
    print(f'  Representativas (más cercanas al centroide):')
    for _, r in reps.iterrows():
        print(f'    - {r["nombre_zona"]} ({r["municipio"]}): '
              f'{r["total_siniestros"]} sin, {r["num_empresas"]} emp')

print(f'\n✓ CHECKPOINT 5: Clusters caracterizados')

## 7. Guardar Resultados y Mapa Folium

In [ ]:
# Guardar CSV con TODAS las 183 zonas (las filtradas con cluster_zona = -1)
df_all = df_zonas.copy()
df_all['cluster_zona'] = -1  # default: sin cluster

# Mapear clusters de las zonas válidas
for _, row in df_valid.iterrows():
    df_all.loc[df_all['zona_id'] == row['zona_id'], 'cluster_zona'] = row['cluster_zona']

# Guardar
output_cols = ['zona_id', 'nombre_zona', 'municipio', 'tipo_zona', 'latitud', 'longitud',
               'num_empresas', 'radio_aprox_m', 'total_siniestros',
               'siniestros_hora_pico', 'siniestros_madrugada', 'siniestros_finde',
               'ratio_lesionados', 'ratio_fallecidos', 'cluster_zona']

df_all[output_cols].to_csv(RUTA_PROCESSED + 'zonas_clusterizadas.csv', index=False)
print(f'✓ CSV guardado: zonas_clusterizadas.csv ({len(df_all)} zonas, '
      f'{(df_all["cluster_zona"]>=0).sum()} con cluster)')

# Mapa folium
colores = {0: 'red', 1: 'blue', 2: 'green', 3: 'purple', 4: 'orange',
           5: 'darkred', 6: 'cadetblue', -1: 'lightgray'}

mapa = folium.Map(location=[25.72, -100.28], zoom_start=11)

for _, row in df_all.iterrows():
    c = int(row['cluster_zona'])
    color = colores.get(c, 'gray')
    radius = max(4, min(18, row['total_siniestros'] / 20))
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=radius, color=color, fill=True, fill_opacity=0.7,
        tooltip=(f"{row['nombre_zona']} | Cluster {c} | "
                 f"{int(row['total_siniestros'])} siniestros | "
                 f"{int(row['num_empresas'])} empresas")
    ).add_to(mapa)

mapa.save(RUTA_OUTPUTS + 'mapa_zonas_clusters.html')
print(f'✓ Mapa guardado: mapa_zonas_clusters.html')

print(f'\n{"="*65}')
print('NB08 COMPLETADO ✓')
print(f'{"="*65}')
for c in sorted(df_all['cluster_zona'].unique()):
    n = (df_all['cluster_zona'] == c).sum()
    label = 'Sin cluster (<5 sin)' if c == -1 else f'Cluster {c}'
    print(f'  {label}: {n} zonas')
print(f'\nSiguiente: 09_Modelo_Clasificacion.ipynb')

In [ ]:
# Renderizar mapa en notebook
mapa